# Framewise Displacement QC

This notebook analyzes framewise displacement (FD) from fMRIPrep confounds files to assess head motion quality across runs.

For each subject and run it:
- Loads the fMRIPrep `desc-confounds_timeseries.tsv`
- Computes FD statistics (mean, max, % of volumes exceeding threshold)
- Generates FD time-series plots with high-motion volumes marked in red
- Saves a standalone HTML QC report per subject

For rule-based outlier detection and motion regressor file generation (used in L1 modeling), see `generate_outlier_report.ipynb`.

## 1. Imports

In [ ]:
import os
import glob
import base64
import warnings
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

## 2. Configuration

All paths are derived from this notebook's location (`scripts/FMRIPREP/`). The only variables you need to change are the subject filter and optional run/task/session filters.

In [ ]:
# ── Subject filter ────────────────────────────────────────────────────────────
# Use 'all' to process every subject, or specify a bare ID like 'GEO037'
SUBJECT = 'all'

# ── FD threshold ──────────────────────────────────────────────────────────────
FD_LIMIT = 0.75   # mm — volumes above this are flagged as high-motion

# ── Optional filters (None = include all) ─────────────────────────────────────
SESSION_FILTER = None   # e.g. 't3' or None
TASK_FILTER    = None   # e.g. 'image' or None
RUN_FILTER     = None   # e.g. '1' or None

# ── Plot settings ─────────────────────────────────────────────────────────────
PLOT_WIDTH  = 12
PLOT_HEIGHT = 6
PLOT_DPI    = 150

# ── Derived paths (do not edit) ───────────────────────────────────────────────
project_dir     = os.path.abspath('../../')
derivatives_dir = os.path.join(project_dir, 'data/bids_data/derivatives_nocorrection')
report_out_dir  = os.path.join(derivatives_dir, 'outlier/fd_reports')
os.makedirs(report_out_dir, exist_ok=True)

print(f'Derivatives  : {derivatives_dir}')
print(f'Reports out  : {report_out_dir}')
print(f'Subject      : {SUBJECT}')
print(f'FD threshold : {FD_LIMIT} mm')

## 3. Helper Functions

In [ ]:
def find_subjects(derivatives_dir, subject_filter):
    """Return list of bare subject IDs (no 'sub-' prefix) to process."""
    base = Path(derivatives_dir)
    if subject_filter.lower() == 'all':
        subjects = sorted([d.name[4:] for d in base.glob('sub-*') if d.is_dir()])
        print(f'Found {len(subjects)} subjects')
    else:
        subjects = [subject_filter]
        if not (base / f'sub-{subject_filter}').exists():
            raise FileNotFoundError(f'Subject directory not found: sub-{subject_filter}')
    return subjects


def find_confounds_files(derivatives_dir, subject_id,
                         session_filter=None, task_filter=None, run_filter=None):
    """Find all confounds TSV files for a subject with optional BIDS filters."""
    subject_dir = Path(derivatives_dir) / f'sub-{subject_id}'
    pattern = f'sub-{subject_id}'
    if session_filter: pattern += f'_ses-{session_filter}'
    if task_filter:    pattern += f'_task-{task_filter}'
    if run_filter:     pattern += f'_run-{run_filter}'
    pattern += '*desc-confounds_timeseries.tsv'
    return list(subject_dir.rglob(pattern))


def parse_bids_filename(filepath):
    """Extract BIDS key-value pairs from a filename."""
    parts = Path(filepath).name.replace('.tsv', '').split('_')
    ids = {}
    for p in parts:
        if '-' in p:
            k, v = p.split('-', 1)
            if k != 'desc':
                ids[k] = v
    return ids


print('Helper functions defined.')

## 4. Load Confounds

In [ ]:
subjects = find_subjects(derivatives_dir, SUBJECT)
all_data = {}  # {subject_id: {run_key: {...}}}

for sub in subjects:
    files = find_confounds_files(derivatives_dir, sub,
                                 SESSION_FILTER, TASK_FILTER, RUN_FILTER)
    if not files:
        print(f'  No confounds files found for sub-{sub} — skipping')
        continue

    sub_data = {}
    for f in sorted(files):
        ids = parse_bids_filename(f)
        run_key = '_'.join(f'sub-{ids.get("sub",sub)}' if k == 'sub'
                           else f'{k}-{v}'
                           for k, v in ids.items())

        df = pd.read_csv(f, sep='\t')
        if 'framewise_displacement' not in df.columns:
            print(f'  WARNING: no framewise_displacement in {f.name}')
            continue

        fd = df['framewise_displacement'].dropna()
        sub_data[run_key] = {'df': df, 'fd': fd, 'file': f, 'ids': ids}
        print(f'  Loaded {run_key}  ({len(df)} volumes)')

    if sub_data:
        all_data[sub] = sub_data

print(f'\nLoaded data for {len(all_data)} subjects.')

## 5. Compute FD Statistics

In [ ]:
results = {}  # {subject_id: {run_key: stats_dict}}
summary_rows = []

for sub, sub_data in all_data.items():
    sub_results = {}
    for run_key, d in sub_data.items():
        fd = d['fd']
        high = fd > FD_LIMIT
        n_high = high.sum()
        pct    = (n_high / len(fd)) * 100

        stats = {
            'fd': fd, 'high_mask': high,
            'n_high': n_high, 'pct_high': pct,
            'mean': fd.mean(), 'median': fd.median(),
            'max': fd.max(), 'std': fd.std(),
            'n_vols': len(fd), 'threshold': FD_LIMIT,
            'file': d['file'], 'ids': d['ids']
        }
        sub_results[run_key] = stats

        # Quality label
        quality = 'GOOD' if pct < 10 else ('MODERATE' if pct < 20 else 'HIGH MOTION')
        flag = '✅' if pct < 10 else ('⚠️' if pct < 20 else '🚨')
        print(f'{flag} sub-{sub}  {run_key:50s}  mean={fd.mean():.3f}  max={fd.max():.3f}  {pct:.1f}% > {FD_LIMIT}mm  [{quality}]')

        summary_rows.append({
            'subject': sub, 'run': run_key,
            'n_volumes': len(fd), 'mean_fd': fd.mean(), 'max_fd': fd.max(),
            'n_high_motion': n_high, 'pct_high_motion': pct, 'quality': quality
        })

    results[sub] = sub_results

summary_df = pd.DataFrame(summary_rows)
print('\nSUMMARY:')
display(summary_df.round(3))

## 6. Plot FD Time Series

In [ ]:
plots_b64 = {}  # store base64 for HTML reports

for sub, sub_results in results.items():
    plots_b64[sub] = {}
    for run_key, s in sub_results.items():
        fig, ax = plt.subplots(figsize=(PLOT_WIDTH, PLOT_HEIGHT))

        ax.plot(s['fd'].values, 'b-', lw=1, alpha=0.8, label='FD')
        if s['high_mask'].any():
            hi_idx = np.where(s['high_mask'])[0]
            ax.scatter(hi_idx, s['fd'].iloc[hi_idx], color='red', s=30, zorder=5,
                       label=f'FD > {FD_LIMIT} mm')
        ax.axhline(FD_LIMIT, color='red', ls='--', alpha=0.7,
                   label=f'Threshold ({FD_LIMIT} mm)')

        stats_text = (f'Mean: {s["mean"]:.3f} mm\nMax: {s["max"]:.3f} mm\n'
                      f'>{FD_LIMIT} mm: {s["n_high"]} vols ({s["pct_high"]:.1f}%)')
        ax.text(0.02, 0.97, stats_text, transform=ax.transAxes, va='top',
                bbox=dict(boxstyle='round', fc='wheat', alpha=0.8), fontsize=10)

        ax.set(xlabel='Volume (TR)', ylabel='FD (mm)',
               title=f'Framewise Displacement — {run_key}')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        buf = BytesIO()
        fig.savefig(buf, format='png', dpi=PLOT_DPI, bbox_inches='tight')
        buf.seek(0)
        plots_b64[sub][run_key] = base64.b64encode(buf.getvalue()).decode()
        plt.close(fig)

## 7. Save HTML QC Reports

Creates a standalone HTML report per subject in `derivatives_nocorrection/outlier/fd_reports/`.

In [ ]:
def make_run_section(plot_b64, s, run_key):
    bg   = '#d4edda' if s['pct_high'] < 10 else ('#fff3cd' if s['pct_high'] < 20 else '#f8d7da')
    fg   = '#155724' if s['pct_high'] < 10 else ('#856404' if s['pct_high'] < 20 else '#721c24')
    qual = 'GOOD'    if s['pct_high'] < 10 else ('MODERATE MOTION' if s['pct_high'] < 20 else 'HIGH MOTION')
    return f"""
    <div style="border:1px solid #ddd;border-radius:8px;padding:20px;margin:20px 0">
      <h2 style="color:#2c3e50;border-bottom:2px solid #3498db;padding-bottom:8px">{run_key}</h2>
      <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:8px;
                  background:#f8f9fa;padding:12px;border-radius:5px;margin-bottom:12px">
        <div><b>Mean FD:</b> {s['mean']:.3f} mm</div>
        <div><b>Median FD:</b> {s['median']:.3f} mm</div>
        <div><b>Max FD:</b> {s['max']:.3f} mm</div>
        <div><b>Std FD:</b> {s['std']:.3f} mm</div>
        <div><b>High-motion vols:</b> {s['n_high']} / {s['n_vols']}</div>
        <div><b>% &gt; {s['threshold']} mm:</b> {s['pct_high']:.1f}%</div>
      </div>
      <div style="padding:10px;border-radius:5px;font-weight:bold;
                  background:{bg};color:{fg};margin-bottom:12px">Quality: {qual}</div>
      <img src="data:image/png;base64,{plot_b64}" style="max-width:100%;border:1px solid #ddd">
      <div style="font-size:11px;color:#666;margin-top:8px">Source: {s['file'].name}</div>
    </div>"""


for sub, sub_results in results.items():
    sections = ''.join(
        make_run_section(plots_b64[sub][rk], s, rk)
        for rk, s in sub_results.items()
    )
    html = f"""<!DOCTYPE html><html><head><meta charset="UTF-8">
    <title>FD Report — sub-{sub}</title>
    <style>body{{font-family:Arial,sans-serif;margin:40px;background:#f5f5f5}}
    .container{{max-width:1200px;margin:0 auto;background:white;padding:30px;
               border-radius:10px;box-shadow:0 2px 10px rgba(0,0,0,.1)}}
    h1{{color:#2c3e50;text-align:center;border-bottom:3px solid #3498db;padding-bottom:12px}}
    </style></head><body><div class="container">
    <h1>Framewise Displacement Report — sub-{sub}</h1>
    <p><b>FD threshold:</b> {FD_LIMIT} mm &nbsp;|&nbsp;
       <b>Generated:</b> {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}</p>
    {sections}
    </div></body></html>"""

    out_path = os.path.join(report_out_dir, f'sub-{sub}_fd_report.html')
    with open(out_path, 'w') as f:
        f.write(html)
    print(f'Saved: {out_path}')

print(f'\nAll HTML reports saved to: {report_out_dir}')

## 8. Save Summary CSV

In [ ]:
csv_path = os.path.join(report_out_dir, 'fd_summary.csv')
summary_df.to_csv(csv_path, index=False)
print(f'Summary saved to: {csv_path}')
display(summary_df.round(3))